In [1]:
import torch
import numpy as numpy
import os
import shutil
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


/home/undergrad/2026/ojonesma/Spring2026/EyeClassifierEnv/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12050). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [2]:
import torch
import sys
print(sys.executable)
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

/home/undergrad/2026/ojonesma/Spring2026/EyeClassifierEnv/bin/python
2.11.0+cu130
13.0
False


Loss function being used is [binary cross entropy with logits loss](https://docs.pytorch.org/docs/1.7.1/generated/torch.nn.BCEWithLogitsLoss.html#torch.nn.BCEWithLogitsLoss). Weights are calculated for the pos_weight parameter for the function. Individual training, and validation, and testing sets both have some instances where a disease is not present. to calculate the weights, a combined table with all diseases is used.

In [4]:
PATH_TRAINING_TRUTHS = "./A. RFMiD_All_Classes_Dataset/2. Groundtruths/a. RFMiD_Training_Labels.csv"
PATH_VALIDATION_TRUTHS = "./A. RFMiD_All_Classes_Dataset/2. Groundtruths/b. RFMiD_Validation_Labels.csv"
PATH_TESTING_TRUTHS = "./A. RFMiD_All_Classes_Dataset/2. Groundtruths/c. RFMiD_Testing_Labels.csv"

# get counts
trainingLabels = pd.read_csv(PATH_TRAINING_TRUTHS)
validationLabels = pd.read_csv(PATH_VALIDATION_TRUTHS)
testingLabels = pd.read_csv(PATH_TESTING_TRUTHS)

total_truths = pd.concat([trainingLabels, validationLabels, testingLabels], ignore_index=True)

# the columns that count the diseases
diseaseCols = total_truths.columns[2:]

#count of all disease incidents, both in total in their own columns as well
counts = total_truths[diseaseCols].sum()
print("Disease Counts")
print(counts)

total_images = total_truths.shape[0]
pos_weights = [(total_images - count)/count for count in counts]
print("pos_weigths for BCEWithLogitsLoss")
print(pos_weights)

Disease Counts
DR      632
ARMD    169
MH      523
DN      230
MYA     167
BRVO    119
TSLN    304
ERM      26
LS       79
MS       27
CSR      61
ODC     445
CRVO     45
TV       10
AH       25
ODP     115
ODE      96
ST       11
AION     26
PT       17
RT       25
RS       71
CRS      54
EDN      24
RPEC     32
MHL      17
RP       10
CWS       8
CB        2
ODPM      2
PRH       5
MNF       3
HR        1
CRAO      4
TD        9
CME       7
PTCR      6
CF        6
VH        4
MCA       1
VS        4
BRAO      4
PLQ       2
HPED      1
CL        2
dtype: int64
pos_weigths for BCEWithLogitsLoss
[4.063291139240507, 17.93491124260355, 5.118546845124283, 12.91304347826087, 18.161676646706585, 25.89075630252101, 9.526315789473685, 122.07692307692308, 39.50632911392405, 117.51851851851852, 51.459016393442624, 6.191011235955056, 70.11111111111111, 319.0, 127.0, 26.82608695652174, 32.333333333333336, 289.90909090909093, 122.07692307692308, 187.23529411764707, 127.0, 44.070422535211264, 58.259